# Task 4: Custom optimizers (Momentum, RMSprop, Adam) with raw PyTorch tensors

In [1]:
import torch

torch.manual_seed(0)

def make_data(n=200):
    X = torch.randn(n, 2)
    y = (X[:,0]**2 + X[:,1]**2 > 1).float().unsqueeze(1)
    return X, y

def model_forward(X, W1, b1, W2, b2):
    h = torch.relu(X @ W1 + b1)
    return torch.sigmoid(h @ W2 + b2)

def bce(out, y):
    return -(y*torch.log(out+1e-9) + (1-y)*torch.log(1-out+1e-9)).mean()


In [2]:
def sgd_momentum(params, grads, velocity, lr=0.1, beta=0.9):
    for p, g, v in zip(params, grads, velocity):
        v.mul_(beta).add_(g)
        p -= lr * v

def rmsprop(params, grads, sq_avg, lr=0.01, beta=0.9, eps=1e-8):
    for p, g, s in zip(params, grads, sq_avg):
        s.mul_(beta).addcmul_(g, g, value=1-beta)
        p -= lr * g / (s.sqrt() + eps)

def adam(params, grads, m, v, t, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
    for p, g, mi, vi in zip(params, grads, m, v):
        mi.mul_(b1).add_(g, alpha=1-b1)
        vi.mul_(b2).addcmul_(g, g, value=1-b2)
        mhat = mi/(1-b1**t)
        vhat = vi/(1-b2**t)
        p -= lr * mhat/(vhat.sqrt()+eps)


In [3]:
X, y = make_data()

def train(opt_name, steps=300):
    W1 = torch.randn(2,16, requires_grad=True)
    b1 = torch.zeros(16, requires_grad=True)
    W2 = torch.randn(16,1, requires_grad=True)
    b2 = torch.zeros(1, requires_grad=True)
    params = [W1,b1,W2,b2]
    velocity = [torch.zeros_like(p) for p in params]
    sq_avg = [torch.zeros_like(p) for p in params]
    m = [torch.zeros_like(p) for p in params]
    v = [torch.zeros_like(p) for p in params]
    losses = []
    for t in range(1, steps+1):
        out = model_forward(X, *params)
        loss = bce(out, y)
        for p in params:
            if p.grad is not None:
                p.grad.zero_()
        loss.backward()
        grads = [p.grad.detach().clone() for p in params]
        with torch.no_grad():
            if opt_name == "momentum":
                sgd_momentum(params, grads, velocity)
            elif opt_name == "rmsprop":
                rmsprop(params, grads, sq_avg)
            elif opt_name == "adam":
                adam(params, grads, m, v, t)
        losses.append(loss.item())
    return losses


In [4]:
for name in ["momentum", "rmsprop", "adam"]:
    losses = train(name)
    print(name, "final loss:", losses[-1])


momentum final loss: 0.054721198976039886
rmsprop final loss: 0.04324650391936302
adam final loss: 0.0822715312242508
